# 08 — Daga et al. (2026): adaptive quantum-kernel stacking, on all 4 diabetes datasets

**Method (Sci. Rep. 16:26840):** QSVMs with Angle, ZZ and Amplitude fidelity kernels, a classical
RBF-SVM, hard voting, and an out-of-fold (OOF) stacked meta-learner (logistic regression) over the
kernel models' decision scores. Original data: Pima only.

**Protocols**
* `paper` — the released script's protocol: scaling fitted on **all rows** (global), ungrouped
  stratified 5-fold CV (seed 42); pooled OOF predictions get a bootstrap CI (as in Table 9).
* `clean` — leakage-free: everything fitted inside the outer training fold; grouped CV when
  feature vectors repeat; seeds {0, 1, 42}. Stacking meta-features come from an inner 5-fold CV
  on the training fold only.

Quantum kernels are exact noiseless simulations (`nb2_common.py`); q = min(d, 8) qubits, PCA
fitted in-fold when d > 8. Positive class per dataset is printed by `load_all()`.

In [1]:
import os, sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd() / "notebooks2", Path("/content/notebooks2"), *Path("/kaggle/input").glob("*"), *Path("/kaggle/input").glob("*/notebooks2")]:
    if (p / "nb2_common.py").exists():
        sys.path.insert(0, str(p)); break
from nb2_common import *
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
sanity_check_kernels()
DATA = load_all()

nb2_common loaded: numpy 2.4.6, sklearn 1.6.1
kernel self-test passed
pima     N=  768 d=  8 prevalence=0.349 unique=  768 grouped_cv=False  [diabetes]
sylhet   N=  520 d= 16 prevalence=0.615 unique=  251 grouped_cv=True  [diabetes (early stage)]


cdc      N= 2000 d= 21 prevalence=0.140 unique= 1995 grouped_cv=True  [diabetes or prediabetes (self-reported)]


hosp130  N= 2000 d= 44 prevalence=0.088 unique= 2000 grouped_cv=False  [readmitted within 30 days]


In [2]:
KERNEL_MODELS = ["Angle-QSVM", "ZZ-QSVM", "Amplitude-QSVM", "RBF-SVM"]


def kernel_blocks(Xq_a, Xq_b, Xs_a, Xs_b):
    """Gram blocks between row sets a and b for the three quantum kernels."""
    return {"Angle-QSVM": angle_kernel(Xq_a, Xq_b),
            "ZZ-QSVM": zz_kernel(Xq_a, Xq_b, reps=2),
            "Amplitude-QSVM": amplitude_kernel(Xs_a, Xs_b)}


def fit_scores(Ktr, Kte, Xs_tr, Xs_te, ytr, seed):
    """Fit the 4 kernel SVMs; return {model: (pred, decision score)} on the test block."""
    out = {m: svm_precomputed(Ktr[m], ytr, Kte[m], C=1.0, seed=seed) for m in Ktr}
    rbf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=seed).fit(Xs_tr, ytr)
    out["RBF-SVM"] = (rbf.predict(Xs_te), rbf.decision_function(Xs_te))
    return out


def daga_method(Xtr, ytr, Xte, seed, groups_tr, X_prep):
    Xq_tr, Xq_te = prep(to_qubits, X_prep, Xtr, Xte, seed=seed)
    Xs_tr, Xs_te = prep(impute_scale, X_prep, Xtr, Xte)
    K_trtr = kernel_blocks(Xq_tr, Xq_tr, Xs_tr, Xs_tr)
    K_tetr = kernel_blocks(Xq_te, Xq_tr, Xs_te, Xs_tr)
    out = fit_scores(K_trtr, K_tetr, Xs_tr, Xs_te, ytr, seed)

    # OOF stacking: meta-features = inner-CV decision scores of the 4 kernel models (train fold only)
    meta_tr = np.zeros((len(ytr), len(KERNEL_MODELS)))
    for itr, iva in inner_splits(ytr, groups_tr, seed, n_splits=5):
        Ka = {m: K_trtr[m][np.ix_(itr, itr)] for m in K_trtr}
        Kb = {m: K_trtr[m][np.ix_(iva, itr)] for m in K_trtr}
        o = fit_scores(Ka, Kb, Xs_tr[itr], Xs_tr[iva], ytr[itr], seed)
        meta_tr[iva] = np.column_stack([o[m][1] for m in KERNEL_MODELS])
    meta_te = np.column_stack([out[m][1] for m in KERNEL_MODELS])
    meta = LogisticRegression(max_iter=1000).fit(meta_tr, ytr)
    out["OOF-Stacking (LR meta)"] = (meta.predict(meta_te), meta.predict_proba(meta_te)[:, 1])

    votes = np.column_stack([out[m][0] for m in KERNEL_MODELS])
    out["Hard-Voting (4 kernels)"] = ((votes.mean(1) > 0.5).astype(int), votes.mean(1))

    lr = LogisticRegression(max_iter=1000).fit(Xs_tr, ytr)
    out["Logistic Regression"] = (lr.predict(Xs_te), lr.predict_proba(Xs_te)[:, 1])
    rf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1).fit(Xs_tr, ytr)
    out["Random Forest"] = (rf.predict(Xs_te), rf.predict_proba(Xs_te)[:, 1])
    out["Majority class"] = (np.full(len(Xte), int(ytr.mean() >= 0.5)), np.zeros(len(Xte)))
    return out

In [3]:
rec = Recorder("P08_Daga", "08_daga", config=dict(
    method="Angle/ZZ(reps2,linear)/Amplitude QSVM C=1, RBF-SVM C=1, inner-5-fold OOF LR stacking, hard voting",
    paper_protocol="global scaling + ungrouped stratified 5-fold seed 42"))
print("paper protocol"); run_paper_cv(rec, DATA, daga_method, global_prep=True)
print("clean protocol"); run_clean(rec, DATA, daga_method)
df = rec.save()

paper protocol


  paper  pima     done


  paper  sylhet   done


  paper  cdc      done


  paper  hosp130  done
clean protocol


  clean  pima     done in   18.7s


  clean  sylhet   done in   16.0s


  clean  cdc      done in   53.3s


  clean  hosp130  done in   50.8s
saved /content/notebooks2/results/08_daga.csv  (720 rows)


## Results: mean ± SD over folds (and seeds for `clean`)

In [4]:
pd.set_option("display.width", 220, "display.max_rows", 200)
summarise(df)

,dataset,protocol,model,accuracy,balanced_accuracy,recall,specificity,roc_auc,n_runs
0,cdc,clean,Amplitude-QSVM,0.86 ± 0.018,0.5 ± 0.001,0.0 ± 0.0,1.0 ± 0.001,0.692 ± 0.031,15
1,cdc,clean,Angle-QSVM,0.86 ± 0.018,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.717 ± 0.034,15
2,cdc,clean,Hard-Voting (4 kernels),0.86 ± 0.018,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.534 ± 0.018,15
3,cdc,clean,Logistic Regression,0.863 ± 0.017,0.577 ± 0.018,0.18 ± 0.039,0.974 ± 0.006,0.818 ± 0.02,15
4,cdc,clean,Majority class,0.86 ± 0.018,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.5 ± 0.0,15
5,cdc,clean,OOF-Stacking (LR meta),0.863 ± 0.016,0.558 ± 0.017,0.133 ± 0.039,0.982 ± 0.008,0.756 ± 0.03,15
6,cdc,clean,RBF-SVM,0.862 ± 0.017,0.531 ± 0.018,0.071 ± 0.039,0.99 ± 0.005,0.75 ± 0.037,15
7,cdc,clean,Random Forest,0.862 ± 0.014,0.551 ± 0.018,0.12 ± 0.038,0.983 ± 0.006,0.801 ± 0.024,15
8,cdc,clean,ZZ-QSVM,0.86 ± 0.018,0.506 ± 0.006,0.014 ± 0.012,0.998 ± 0.002,0.684 ± 0.029,15
9,cdc,paper,Amplitude-QSVM,0.86 ± 0.001,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.659 ± 0.04,5


## Pooled out-of-fold accuracy and recall (paper vs clean), stacking vs RBF vs Angle
Bootstrap CIs and significance tests are computed in `09_summary.ipynb`.
Hard voting over 4 models counts a 2–2 tie as negative.

In [5]:
rows = []
for (dset, proto, seed, model), g in df.groupby(["dataset", "protocol", "seed", "model"]):
    if model not in ("OOF-Stacking (LR meta)", "RBF-SVM", "Angle-QSVM"):
        continue
    # rebuild pooled confusion counts -> accuracy CI via per-fold sums
    acc = (g.tp.sum() + g.tn.sum()) / g.n_test.sum()
    rec_ = g.tp.sum() / max(g.tp.sum() + g.fn.sum(), 1)
    rows.append(dict(dataset=dset, protocol=proto, seed=seed, model=model, pooled_acc=round(acc, 4),
                     pooled_recall=round(rec_, 4)))
pd.DataFrame(rows).pivot_table(index=["dataset", "model"], columns="protocol",
                               values=["pooled_acc", "pooled_recall"], aggfunc="mean").round(4)

pooled_acc         pooled_recall        
protocol                            clean   paper         clean   paper
dataset model                                                          
cdc     Angle-QSVM                 0.8605  0.8605        0.0000  0.0000
        OOF-Stacking (LR meta)     0.8635  0.8645        0.1302  0.1111
        RBF-SVM                    0.8617  0.8625        0.0693  0.0573
hosp130 Angle-QSVM                 0.9120  0.9120        0.0000  0.0000
        OOF-Stacking (LR meta)     0.9120  0.9120        0.0000  0.0000
        RBF-SVM                    0.9120  0.9120        0.0000  0.0000
pima    Angle-QSVM                 0.7591  0.7578        0.5522  0.5560
        OOF-Stacking (LR meta)     0.7613  0.7591        0.5672  0.5597
        RBF-SVM                    0.7595  0.7604        0.5560  0.5597
sylhet  Angle-QSVM                 0.8872  0.9481        0.9188  0.9500
        OOF-Stacking (LR meta)     0.9243  0.9635        0.9302  0.9625
        RBF-SVM                    0.9282  0.9731        0.9417  0.9781